In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, sum as spark_sum, count, desc, rank, max as spark_max, year, current_date)
from pyspark.sql.window import Window
from pyspark.sql import functions as F


spark = SparkSession.builder.getOrCreate()

In [0]:
# Receita total em R$

# Aqui pega-se a coluna receita_brl da tabela fato, que já foi convertida usando a cotação do dólar que puxamos da API do Banco Central
# Usa-se o sum() para somar tudo e o round() para deixar com 2 casas decimais

df_q1 = spark.sql("""
SELECT
    ROUND(SUM(receita_brl), 2) AS receita_total_brl
FROM gold.fact_movies_performance
""")

display(df_q1)

receita_total_brl
903505260414.58


In [0]:
# Top 5 filmes por popularidade

# Junta-se a dim_movies com a fact_movies_performance para trazer o título e a nota de popularidade
# Ordena-se do maior para o menor e limita em 5

df_q2 = spark.sql("""
SELECT
    d.titulo,
    f.popularidade
FROM gold.fact_movies_performance f
JOIN gold.dim_movies d
    ON f.sk_movie_id = d.sk_movie_id
ORDER BY f.popularidade DESC
LIMIT 5
""")

display(df_q2)

# Nota: Os filmes La Fellinette (2020), The Fear Footage 2 (2019) e Battipaglia 1969 (1969) apresentam valores de popularidade identificados como provenientes de Column Shift na base bruta. Esses valores foram considerados inconsistentes e, conforme as regras de tratamento da camada Silver, devem ser tratados como `NULL`. Desconsiderando esses valores no ranking, os cinco filmes com maior popularidade são: Blue Beetle, Gran Turismo, The Nun II, Meg 2: The Trench e Retribution.


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


In [0]:
# Quantidade de filmes por gênero
# Como um filme pode ter mais de um gênero, usa-se a tabela bridge (bridge_movie_genre) para ligar a dimensão de filmes com a dim_genres sem duplicar dados
# Agrupa-se pelo nome do gênero e faz a contagem de quantos filmes distintos cada um tem

df_q3 = spark.sql("""
SELECT
    g.nome_genero,
    COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
FROM gold.bridge_movie_genre b
JOIN gold.dim_genres g
    ON b.sk_genre_id = g.sk_genre_id
GROUP BY g.nome_genero
ORDER BY qtd_filmes DESC
""")

display(df_q3)

nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
Tv Movie,4066


In [0]:
# Top 10 filmes com maior receita
# Conforme o enunciado pediu, cria-se uma Window ordenada pela receita em dólar descrescente para ranquear os filmes de 1 a 10
# Pega-se tanto a receita em US$ quanto o valor convertido em R$

df_q4 = spark.sql("""
SELECT
    d.titulo,
    f.receita_usd,
    f.receita_brl,
    RANK() OVER (ORDER BY f.receita_usd DESC) AS ranking
FROM gold.fact_movies_performance f
JOIN gold.dim_movies d
    ON f.sk_movie_id = d.sk_movie_id
WHERE f.receita_usd IS NOT NULL
ORDER BY f.receita_usd DESC
LIMIT 10
""")

display(df_q4)

titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,13915440000.00,1
Avatar: The Way of Water,2320250281.00,11531179846.51,2
AVENGERS: INFINITY WAR,2052415039.00,10200092260.82,3
spider-man: no way home,1921847111.00,9551195772.25,4
The Lion King,1663075401.00,8265152127.89,5
Top Gun: Maverick,1488732821.00,7398704373.81,6
Barbie,1428545028.00,7099583080.15,7
The Super Mario Bros. Movie,1355725263.00,6737683412.06,8
Black Panther,1349926083.00,6708862647.29,9
Star Wars: The Last Jedi,1332698830.00,6623246645.33,10


In [0]:
# Pega-se a data mais recente da base que já foi lançada e que não seja data futura. A partir dela, volta-se 2 anos
# Depois, junta-se os filmes dessa janela com a bridge de pessoas e a dim_people, filtrando só quem é 'Ator' e ordenando pelo maior número de participações

# Definir a data máxima realizada (status 'Lançado' e ignorando datas futuras além de hoje)

df_filmes_validos = (
    spark.table("gold.dim_movies")
    .filter(
        (F.col("status_filme") == "Lançado") & 
        (F.col("data_lancamento") <= F.current_date())
    )
)

max_data = df_filmes_validos.select(F.max("data_lancamento")).first()[0]

# Janela dos últimos 2 anos (subtrair 2 anos da data máxima realizada)
data_inicio_2_anos = F.add_months(F.lit(max_data), -24)

# Filtrar os filmes dentro dessa janela
filmes_2_anos = df_filmes_validos.filter(
    (F.col("data_lancamento") >= data_inicio_2_anos) &
    (F.col("data_lancamento") <= F.lit(max_data))
)

# Juntar com as tabelas de pessoas para encontrar o ator com mais participações
ator_top = (
    filmes_2_anos
    .join(spark.table("gold.bridge_movie_person"), on="sk_movie_id", how="inner")
    .join(spark.table("gold.dim_people"), on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator") # Apenas atores
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc()) # Ordenar pelo maior número de filmes
    .limit(1)
)

display(ator_top)


nome_pessoa,qtd_filmes
Kevin Hart,64


In [0]:
# Mesma lógica da regra de datas: pega-se o ano máximo de filmes lançados até a data atual e subtrai 5 anos para fechar a janela
# Em seguida, junta-se os filmes com a fact_movies_performance (para pegar o lucro em R$) e com a bridge de empresas para somar o lucro por produtora e pegar a primeira colocada

# Obter o ano máximo de filmes lançados e válidos
ano_max = (
    spark.table("gold.dim_movies")
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .select(F.max("ano_lancamento"))
    .first()[0]
)

ano_min = ano_max - 5

# Filtrar filmes dos últimos 5 anos
df_filmes_5anos = (
    spark.table("gold.dim_movies")
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("ano_lancamento") >= ano_min) &
        (F.col("ano_lancamento") <= ano_max)
    )
    .select("sk_movie_id")
)

# Consulta final juntando Fato e Dimensões
df_produtora_top = (
    df_filmes_5anos
    .join(spark.table("gold.fact_movies_performance").select("sk_movie_id", "lucro_brl"), on="sk_movie_id")
    .join(spark.table("gold.bridge_movie_company"), on="sk_movie_id")
    .join(spark.table("gold.dim_companies"), on="sk_company_id")
    .filter(F.col("lucro_brl").isNotNull())
    .groupBy("nome_produtora")
    .agg(F.round(F.sum("lucro_brl"), 2).alias("lucro_total_brl"))
    .orderBy(F.col("lucro_total_brl").desc())
    .limit(1)
)

display(df_produtora_top)


nome_produtora,lucro_total_brl
Universal Pictures,29221514099.19
